# Data Exploration and Preparation

**EDA notebook lead:** Aura Gaines  
**Project team:** Alan Jiang, Aura Gaines, Brandon Shumack, and Fil Dziembowski

In [1]:
# Install the packages we need for this notebook.
# datasets = lets us load the Hugging Face dataset
# pyarrow = lets pandas read/write parquet files
!pip install datasets pyarrow scikit-learn pandas

In [2]:
# import packages
import pandas as pd
import numpy as np
import json
import os

# Keep Hugging Face download progress bars from being saved as widget outputs.
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from datasets import load_dataset
from sklearn.model_selection import train_test_split

/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# set seed and output folder
SEED = 42
OUTPUT_DIR = "data_splits"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# load dataset from hugging face
dataset = load_dataset("ai4privacy/pii-masking-400k", split="train")

# turn dataset into a pandas dataframe
df = dataset.to_pandas()

# check size and columns
print(len(df))
print(df.columns.tolist())

325517
['source_text', 'locale', 'language', 'split', 'privacy_mask', 'uid', 'masked_text', 'mbert_tokens', 'mbert_token_classes']


In [5]:
# show all column names
df.columns.tolist()

['source_text',
 'locale',
 'language',
 'split',
 'privacy_mask',
 'uid',
 'masked_text',
 'mbert_tokens',
 'mbert_token_classes']

In [6]:
# look at first few rows
df.head()

,source_text,locale,language,split,privacy_mask,uid,masked_text,mbert_tokens,mbert_token_classes
0,<p>My child faozzsd379223 (DOB: May/58) will u...,US,en,train,"[{'label': 'USERNAME', 'start': 12, 'end': 25,...",302521,<p>My child [USERNAME_2] (DOB: [DATEOFBIRTH_1]...,"[<, p, >, My, child, fa, ##oz, ##zs, ##d, ##3,...","[O, O, O, O, O, B-USERNAME, I-USERNAME, I-USER..."
1,Guardians:*BF6* and *BF6* grant permission for...,GB,en,train,"[{'label': 'USERNAME', 'start': 11, 'end': 14,...",120409,Guardians:*[USERNAME_4]* and *[USERNAME_3]* gr...,"[Guardian, ##s, :, *, BF, ##6, *, and, *, BF, ...","[O, O, O, O, B-USERNAME, I-USERNAME, O, O, O, ..."
2,"We, *bahara.cathers19* and *bahara.cathers19* ...",GB,en,train,"[{'label': 'USERNAME', 'start': 5, 'end': 21, ...",120411,"We, *[USERNAME_3]* and *[USERNAME_2]* reside a...","[We, ,, *, ba, ##hara, ., cat, ##hers, ##19, *...","[O, O, O, B-USERNAME, I-USERNAME, I-USERNAME, ..."
3,Student: Blagojka van der Boog\nDOB: 8th Janua...,US,en,train,"[{'label': 'GIVENNAME', 'start': 9, 'end': 17,...",128429,Student: [GIVENNAME_2] [SURNAME_2]\nDOB: [DATE...,"[Student, :, B, ##lag, ##oj, ##ka, van, der, B...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ..."
4,Child: Anna-Louise Dolderer\nDate of Birth: 05...,US,en,train,"[{'label': 'GIVENNAME', 'start': 7, 'end': 18,...",128431,Child: [GIVENNAME_2] [SURNAME_2]\nDate of Birt...,"[Child, :, Anna, -, Louise, Dol, ##dere, ##r, ...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ..."


**Dataset inspection notes**

The dataset has original text, masked text, privacy masks, and token-level PII labels. The main input text appears to be source_text. The token-level PII labels are in mbert_token_classes.

In [7]:
# look at one row clearly
row = df.iloc[0]

print("source text:")
print(row['source_text'])

print("\nprivacy mask:")
print(row['privacy_mask'])

print("\nmbert token classes:")
print(row['mbert_token_classes'])

source text:
<p>My child faozzsd379223 (DOB: May/58) will undergo treatment with Dr. faozzsd379223, office at Hill Road. Our ZIP code is 28170-6392. Consult policy M.UE.227995. Contact number: 0070.606.322.6244. Handle transactions with 6225427220412963. Queries? Email: faozzsd379223@outlook.com.</p>

privacy mask:
[{'label': 'USERNAME', 'start': 12, 'end': 25, 'value': 'faozzsd379223', 'label_index': 2}
 {'label': 'DATEOFBIRTH', 'start': 32, 'end': 38, 'value': 'May/58', 'label_index': 1}
 {'label': 'USERNAME', 'start': 72, 'end': 85, 'value': 'faozzsd379223', 'label_index': 1}
 {'label': 'STREET', 'start': 97, 'end': 106, 'value': 'Hill Road', 'label_index': 1}
 {'label': 'ZIPCODE', 'start': 124, 'end': 134, 'value': '28170-6392', 'label_index': 1}
 {'label': 'TELEPHONENUM', 'start': 180, 'end': 197, 'value': '0070.606.322.6244', 'label_index': 1}
 {'label': 'CREDITCARDNUMBER', 'start': 224, 'end': 240, 'value': '6225427220412963', 'label_index': 1}
 {'label': 'EMAIL', 'start': 258, 

**Single-row inspection notes**

This example shows how the dataset stores PII. The `source_text` column contains the original text. The `privacy_mask` column lists the PII categories and values found in the text. The `mbert_token_classes` column gives token-level labels, where `O` means not PII and labels like `B-EMAIL` or `I-EMAIL` mark PII tokens.

For the main binary task, any row with a token label other than `O` should be labeled privacy-sensitive.

In [8]:
# make binary safe/pii label
def is_sensitive(row):
    tags = row.get('mbert_token_classes', None)

    if tags is not None:
        return int(any(str(t).strip() != 'O' for t in tags))

    return 0

# apply label function to every row
df['label'] = df.apply(is_sensitive, axis=1)

In [9]:
# check label counts
df['label'].value_counts()

label
1    219260
0    106257
Name: count, dtype: int64

In [10]:
# check label percentages
df['label'].value_counts(normalize=True) * 100

label
1    67.357465
0    32.642535
Name: proportion, dtype: float64

**Binary label check**

Using `mbert_token_classes`, I created a binary label where any token label other than `O` becomes `1` = `privacy-sensitive`. Rows with only `O` labels become `0 = safe`.

The resulting class balance is about 67.4% sensitive and 32.6% safe, which matches the split metadata. This confirms that the notebook is using the same label-collapse logic as `src/data/data_split.py`.

In [11]:
# keep only columns needed for main binary task
df_core = df[['source_text', 'label']].copy()

# rename source_text to text so it is easier to use later
df_core.columns = ['text', 'label']

# keep original row number so we can connect back to pii categories later
df_core['original_index'] = df.index

# drop rows where text is missing
df_core = df_core.dropna(subset=['text'])

# check the cleaned dataframe
df_core.head()

,text,label,original_index
0,<p>My child faozzsd379223 (DOB: May/58) will u...,1,0
1,Guardians:*BF6* and *BF6* grant permission for...,1,1
2,"We, *bahara.cathers19* and *bahara.cathers19* ...",1,2
3,Student: Blagojka van der Boog\nDOB: 8th Janua...,1,3
4,Child: Anna-Louise Dolderer\nDate of Birth: 05...,1,4


**Clean modeling dataframe**

I created a smaller dataframe for the main binary classification task. It keeps only the prompt text, the binary label, and the original row index. The original_index column is important because it lets us reconnect back to the full dataset later for PII category analysis.

In [12]:
# make train and temp split
train_df, temp_df = train_test_split(
    df_core,
    test_size=0.20,
    random_state=SEED,
    stratify=df_core['label']
)

# split temp into validation and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['label']
)

In [13]:
# check split sizes
print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)

train: (260413, 3)
validation: (32552, 3)
test: (32552, 3)


### Initial stratified split check (before cleaning)

As an early EDA check, I recreated an 80/10/10 stratified split with seed 42 before removing blank and duplicate text. The initial split sizes were:

- train: 260,413 rows
- validation: 32,552 rows
- test: 32,552 rows

This confirmed that the split logic preserved class balance, but the later duplicate-overlap check showed that the raw row-level split should not be used as the final frozen split. The project therefore cleaned the data before regenerating the final splits in `src/data/data_split.py`.


In [14]:
# check class balance for each split
for name, split_df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(name)
    print(split_df['label'].value_counts(normalize=True) * 100)
    print()

train
label
1    67.35762
0    32.64238
Name: proportion, dtype: float64

validation
label
1    67.356844
0    32.643156
Name: proportion, dtype: float64

test
label
1    67.356844
0    32.643156
Name: proportion, dtype: float64



### Initial split class balance check

The initial train, validation, and test splits kept nearly the same class balance: about 67.4% privacy-sensitive and 32.6% safe. This confirmed that stratification worked correctly, so the same stratified approach was retained when the final cleaned split was generated.


In [15]:
# check overlap between splits
train_texts = set(train_df['text'])
val_texts = set(val_df['text'])
test_texts = set(test_df['text'])

print("train and validation overlap:", len(train_texts.intersection(val_texts)))
print("train and test overlap:", len(train_texts.intersection(test_texts)))
print("validation and test overlap:", len(val_texts.intersection(test_texts)))

train and validation overlap: 11
train and test overlap: 9
validation and test overlap: 3


In [16]:
# check duplicate text in full dataset
duplicate_count = df_core['text'].duplicated().sum()

print("duplicate text rows:", duplicate_count)
print("unique text rows:", df_core['text'].nunique())
print("total rows:", len(df_core))

duplicate text rows: 93
unique text rows: 325424
total rows: 325517


In [17]:
# look at duplicate rows without printing full text
duplicate_rows = df_core[df_core['text'].duplicated(keep=False)]

duplicate_rows[['label', 'original_index']].head(10)

,label,original_index
54284,0,54284
54299,0,54299
115365,0,115365
115366,0,115366
115368,0,115368
115369,0,115369
115370,0,115370
115372,0,115372
115373,0,115373
115375,0,115375


### Duplicate text check

The raw dataset contained 93 duplicate text rows after the first copy. In the initial row-level split, a small number of exact duplicate texts crossed train, validation, and test boundaries.

This finding led the project to remove blank text and exact duplicate text before generating the final frozen split. The final cleaned split contains no exact duplicate text overlap across train, validation, and test.


In [18]:
# add simple text length columns
df_core['char_count'] = df_core['text'].str.len()
df_core['word_count'] = df_core['text'].str.split().str.len()

In [19]:
# compare text length by label
df_core.groupby('label')[['char_count', 'word_count']].describe()

char_count                                                            \
           count        mean        std   min    25%    50%    75%     max   
label                                                                        
0       106257.0  133.074734  55.241692   0.0   95.0  122.0  158.0   854.0   
1       219260.0  157.078934  60.776068  14.0  113.0  145.0  191.0  1377.0   

      word_count                                                     
           count       mean       std  min   25%   50%   75%    max  
label                                                                
0       106257.0  17.536040  6.707703  0.0  13.0  17.0  21.0   62.0  
1       219260.0  19.837417  7.197647  1.0  15.0  19.0  24.0  103.0

**Main observation**

Privacy-sensitive examples are slightly longer on average than safe examples.

- Safe examples average about 133 characters and 17.5 words.
- Sensitive examples average about 157 characters and 19.8 words.

This means text length may be somewhat related to the label, but the difference is not large enough to rely on by itself.

In [20]:
# Check rows where the text is empty or missing
empty_text_rows = df_core[df_core['char_count'] == 0]

# Show how many empty-text rows we found
print("Number of empty text rows:", len(empty_text_rows))

# Display a few empty-text rows so we can inspect them
empty_text_rows.head()

Number of empty text rows: 6


,text,label,original_index,char_count,word_count
116057,,0,116057,0,0
116068,,0,116068,0,0
120174,,0,120174,0,0
215363,,0,215363,0,0
316042,,0,316042,0,0


### Empty text rows

The raw dataset contained 6 rows where the text field was blank. All 6 were labeled safe (`label = 0`) and contained no useful language features. The final preprocessing pipeline removed these rows before generating the frozen split.


In [21]:
# Sort the dataset by longest text first
longest_rows = df_core.sort_values(by='char_count', ascending=False)

# Look at the longest examples
longest_rows[['text', 'label', 'char_count', 'word_count']].head()

,text,label,char_count,word_count
25452,"<form>\n<label for=""name"">Child's Name:</label...",1,1377,101
172441,<form>\n<label for='prenom'>Prénom :</label>\n...,1,1230,103
25453,"<form>\n<label for=""childName"">Child's Name:</...",1,1121,80
25454,"<form>\n<label for=""pTitle"">Guardian's Mrs:</l...",1,1059,73
94299,<form action='submit_lactation_form' method='p...,1,1012,70


**Longest text rows**

The longest examples are mostly labeled privacy-sensitive and appear to contain HTML/form-like text. This makes sense because forms often include fields such as names, child names, guardian names, or other personal information. This also means the model may learn patterns from structured/form text, not only normal conversational text.

A limitation is that some privacy-sensitive examples are structured forms or HTML-like text. Models may partly learn formatting patterns associated with forms, rather than only detecting PII content itself.

In [22]:
# Count how many examples are in each label
label_counts = df_core['label'].value_counts().sort_index()

# Convert label counts into percentages
label_percentages = df_core['label'].value_counts(normalize=True).sort_index() * 100

# Combine counts and percentages into one table
label_summary = pd.DataFrame({
    'count': label_counts,
    'percentage': label_percentages
})

# Display the label summary table
label_summary

,count,percentage
label,,
0,106257,32.642535
1,219260,67.357465


**Label distribution**

The dataset is imbalanced. There are 106,257 safe examples (`label = 0`), which is about 32.6% of the dataset. There are 219,260 privacy-sensitive examples (`label = 1`), which is about 67.4% of the dataset.

This matters because accuracy alone could be misleading. A model could perform well overall by favoring the majority class, so later evaluation should include precision, recall, and F1 score, especially for the privacy-sensitive class.

In [23]:
# Check how many missing values are in each column
missing_values = df_core.isnull().sum()

# Display the missing value counts
missing_values

text              0
label             0
original_index    0
char_count        0
word_count        0
dtype: int64

### Missing values

There are no missing/null values in the current working columns (`text`, `label`, `original_index`, `char_count`, and `word_count`). This means the dataset does not have `NaN` values in these fields.

However, this is different from empty strings. Earlier, I found 6 rows where `text` is an empty string, even though those rows are not technically missing/null values.

In [24]:
# Count how many duplicated text values exist in the dataset
num_duplicate_texts = df_core['text'].duplicated().sum()

# Print the number of duplicate text rows
print("Number of duplicate text rows:", num_duplicate_texts)

# Look at a few duplicated text rows
duplicate_text_rows = df_core[df_core['text'].duplicated(keep=False)].sort_values(by='text')

# Display a few duplicate rows
duplicate_text_rows[['text', 'label', 'original_index', 'char_count', 'word_count']].head(10)

Number of duplicate text rows: 93


,text,label,original_index,char_count,word_count
116057,,0,116057,0,0
116068,,0,116068,0,0
120174,,0,120174,0,0
215363,,0,215363,0,0
316042,,0,316042,0,0
316043,,0,316043,0,0
129480,2 Behandlungsvorschlag Dokument zur gepl...,0,129480,134,13
129497,2 Behandlungsvorschlag Dokument zur gepl...,0,129497,134,13
211167,</html>,0,211167,7,1
211185,</html>,0,211185,7,1


### Duplicate text rows

The raw dataset contained 93 rows whose `text` value duplicated an earlier row. Some duplicates were blank strings, while others were short structured snippets such as HTML fragments or repeated document phrases.

Because exact duplicates can leak across split boundaries, the final preprocessing pipeline removed blank rows and then deduplicated by exact text before splitting. The regenerated frozen split was verified to have no exact duplicate overlap across train, validation, and test.


In [25]:
# Check the column names currently available in df_core
# We need to see whether this dataframe already knows which rows are train, validation, or test

df_core.columns

Index(['text', 'label', 'original_index', 'char_count', 'word_count'], dtype='str')

In [26]:
# check if any duplicated text has more than one label

duplicate_label_check = (
    df_core[df_core["text"].duplicated(keep=False)]
    .groupby("text")
    .agg(
        num_rows=("text", "size"),
        num_labels=("label", "nunique"),
        labels=("label", lambda x: sorted(x.unique()))
    )
    .sort_values(by="num_labels", ascending=False)
)

duplicate_label_check.head(20)

,num_rows,num_labels,labels
text,,,
,6,1,[0]
2 Behandlungsvorschlag Dokument zur geplanten Behandlung eines Patienten mit angeborenen Herzfehler. Kreditwürdigkeit: Schlecht.,2,1,[0]
</html>,4,1,[0]
"<div class=""participant"">\n <p>Nombre: [NAME_1]<br>\n Edad: [AGE_1]<br>\n Dirección secundaria: [SECONDARYADDRESS_1]<br>\n País: [COUNTRY_1]<br>\n VIN: [VEHICLEVIN_1]<br>\n Tarjeta de identificación: [IDCARDNUM_1]<br>\n Número de tarjeta: [CREDITCARDNUMBER_1]<br>\n Salario: [SALARY_1]<br>\n Fecha de nacimiento: [DATEOFBIRTH_1]</p>\n</div>",2,1,[0]
<div> Sesso: Femmina </div>,2,1,[0]
<html><body><label>Nom complet:</label><input type='text' name='nom_complet' /></body></html>,3,1,[0]
<p><strong>Genere:</strong> Altro,2,1,[0]
<p>CONTACT DETAILS</p><br><label for='address'>Address: [ADDRESS_1]</label><br><label for='phone'>Phone Number: [PHONE_1]</label><br><label for='email'>Email: [EMAIL_1]</label>,8,1,[0]
"<p>Kind: Naam: [FULLNAME_2] Leeftijd: [DATEOFBIRTH_1] Contactpersoon: [FULLNAME_1] Relatie: Grootmoeder Telefoon: [PHONE_1] Adres: [STREET_1], [ZIPCODE_1], Nederland</p>",4,1,[0]


In [27]:
# show duplicate texts where labels disagree

conflicting_duplicate_labels = duplicate_label_check[duplicate_label_check["num_labels"] > 1]

print("Number of duplicate text groups with conflicting labels:", len(conflicting_duplicate_labels))

conflicting_duplicate_labels

Number of duplicate text groups with conflicting labels: 0


,num_rows,num_labels,labels
text,,,


### Duplicate label consistency

No duplicated text appeared under both classes.

This made cleanup straightforward: deduplication removed repeat copies without requiring a choice between conflicting safe and privacy-sensitive labels.

In [28]:
# preview how many rows basic cleaning would remove

blank_text_count = (df_core["text"].fillna("").str.strip() == "").sum()
duplicate_text_count = df_core["text"].duplicated().sum()

df_clean_preview = df_core[df_core["text"].fillna("").str.strip() != ""]
df_clean_preview = df_clean_preview.drop_duplicates(subset=["text"], keep="first")

print("blank text rows:", blank_text_count)
print("duplicate text rows after the first copy:", duplicate_text_count)
print("current total rows:", len(df_core))
print("rows after removing blanks and exact duplicates:", len(df_clean_preview))
print("rows removed:", len(df_core) - len(df_clean_preview))

blank text rows: 6
duplicate text rows after the first copy: 93
current total rows: 325517
rows after removing blanks and exact duplicates: 325423
rows removed: 94


### Cleaning check

Removing blank text and exact duplicate text reduced the dataset from 325,517 to 325,423 rows, a total reduction of 94 rows. The 6 blank rows were removed first; 88 additional duplicate copies remained after that removal.

Based on this EDA result, the final `src/data/data_split.py` workflow removed blank text and exact duplicates before creating the frozen train, validation, and test sets.


In [29]:
# compare label balance before and after the cleaning preview

before_cleaning = df_core["label"].value_counts(normalize=True).sort_index() * 100
after_cleaning = df_clean_preview["label"].value_counts(normalize=True).sort_index() * 100

cleaning_label_comparison = pd.DataFrame({
    "before_cleaning_percent": before_cleaning,
    "after_cleaning_percent": after_cleaning
})

cleaning_label_comparison

,before_cleaning_percent,after_cleaning_percent
label,,
0,32.642535,32.623078
1,67.357465,67.376922


### Cleaning and label balance

The completed cleaning step did not meaningfully change the label distribution. Before cleaning, the dataset was about 32.6% safe and 67.4% privacy-sensitive. After removing blank text and exact duplicates, the percentages remained approximately the same.

This confirms that cleaning reduced leakage risk without materially changing the class balance.


In [30]:
# compare text length by label after the cleaning preview

df_clean_preview.groupby("label")[["char_count", "word_count"]].describe()

char_count                                                            \
           count        mean        std   min    25%    50%    75%     max   
label                                                                        
0       106163.0  133.054087  55.198084   2.0   95.0  122.0  158.0   854.0   
1       219260.0  157.078934  60.776068  14.0  113.0  145.0  191.0  1377.0   

      word_count                                                     
           count       mean       std  min   25%   50%   75%    max  
label                                                                
0       106163.0  17.540348  6.705738  1.0  13.0  17.0  21.0   62.0  
1       219260.0  19.837417  7.197647  1.0  15.0  19.0  24.0  103.0

### Text length after cleaning

After removing blank text and exact duplicates, privacy-sensitive examples remained slightly longer on average than safe examples.

The cleaning step therefore did not change the earlier text-length pattern. Length may provide supporting signal, but it is not sufficient by itself to detect PII.


In [31]:
# look at a few examples from each label

safe_examples = df_clean_preview[df_clean_preview["label"] == 0].sample(5, random_state=42)
sensitive_examples = df_clean_preview[df_clean_preview["label"] == 1].sample(5, random_state=42)

display(safe_examples[["text", "label", "char_count", "word_count"]])
display(sensitive_examples[["text", "label", "char_count", "word_count"]])

,text,label,char_count,word_count
288054,Visita il sito http://www.santoro.org/ per le ...,0,63,8
221745,"Salut, le montant du solde de ce mois est 0.68...",0,81,18
155676,Puedes comprobar las fotos del evento de recic...,0,152,24
222877,<p>Les coordonnées géographiques de votre bure...,0,147,19
48210,Review updates on ENG's health services. Confi...,0,87,12


,text,label,char_count,word_count
305973,"Goededag stoyan.alva, vergeet je volgende afsp...",1,105,14
194615,"Alors Ivett, as-tu eu la chance de consulter h...",1,155,25
236879,"<!DOCTYPE html><html><body><p>Caro Búrik,</p><...",1,277,26
136393,<p>Formulario de aprobación ética para el estu...,1,279,29
33283,"Hello 7-year-old applicant, your auto insuranc...",1,151,21


### Sample examples by label

I looked at a few random examples from each label to sanity check what the classes look like.

The safe examples mostly look like normal text snippets, although some still include URLs or formatting. The privacy-sensitive examples appear more likely to include personal context, forms, HTML-like structure, or placeholders for personal information.

This supports the main project setup, but it also shows a limitation: the model may learn formatting or placeholder patterns in addition to learning what privacy-sensitive text looks like.

In [32]:
# check how often each row has pii style placeholder tokens

df_clean_preview["has_placeholder"] = df_clean_preview["text"].str.contains(r"\[[A-Z]+(?:_[A-Z]+)*_\d+\]", regex=True)

placeholder_summary = (
    df_clean_preview
    .groupby("label")["has_placeholder"]
    .agg(["count", "sum", "mean"])
)

placeholder_summary["percentage"] = placeholder_summary["mean"] * 100

placeholder_summary

,count,sum,mean,percentage
label,,,,
0,106163,3979,0.03748,3.748010
1,219260,3050,0.01391,1.391043


### Placeholder token check

I checked how often examples contain bracket-style placeholder tokens, such as names, emails, phone numbers, or similar structured placeholders.

The placeholder pattern appears in both classes and is actually more common in the safe class than the privacy-sensitive class in this check. This means the label is not simply based on whether the text contains bracketed placeholder tokens.

This is useful because it reduces one concern about the dataset, but it does not remove all formatting concerns. Some examples still include forms, HTML, or structured text that models may learn from.

In [33]:
# check how often text has html-like formatting

df_clean_preview["has_html_like_text"] = df_clean_preview["text"].str.contains(r"<[^>]+>", regex=True)

html_summary = (
    df_clean_preview
    .groupby("label")["has_html_like_text"]
    .agg(["count", "sum", "mean"])
)

html_summary["percentage"] = html_summary["mean"] * 100

html_summary

,count,sum,mean,percentage
label,,,,
0,106163,17915,0.168750,16.874994
1,219260,57101,0.260426,26.042598


### HTML-like text check

HTML-like formatting appears in both classes, but it is more common in the privacy-sensitive examples.

This supports the earlier observation from the longest rows. Some privacy-sensitive examples look like forms or structured text, so the model may learn formatting patterns along with actual PII patterns.

### EDA findings and final project response

This notebook examined the raw AI4Privacy data before modeling. It checked the binary label construction, class balance, initial split behavior, missing and blank text, duplicate rows, text length, representative examples, placeholder tokens, and HTML-like formatting.

#### Main findings

**Binary label setup:**  
Rows with any non-`O` token label in `mbert_token_classes` were labeled privacy-sensitive (`label = 1`); rows containing only `O` labels were labeled safe (`label = 0`). In the raw data, this produced 219,260 privacy-sensitive examples and 106,257 safe examples.

**Label distribution:**  
The raw dataset was approximately 67.4% privacy-sensitive and 32.6% safe. Because of this imbalance, the project reports precision, recall, F1, and confusion-matrix counts rather than relying on accuracy alone.

**Initial split diagnosis:**  
The first 80/10/10 stratified split contained 260,413 training rows, 32,552 validation rows, and 32,552 test rows. Although stratification preserved class balance, exact duplicate text crossed split boundaries: 11 train-validation overlaps, 9 train-test overlaps, and 3 validation-test overlaps. This was an exploratory result, not the final frozen split.

**Blank and duplicate text:**  
The raw dataset contained 6 blank text rows and 93 duplicate rows after the first copy. No duplicated text appeared under conflicting labels. Removing blank text first left 88 additional duplicate copies to remove, for a total reduction of 94 rows.

**Completed cleaning and frozen split:**  
The final preprocessing workflow removed blank text and exact duplicate text before splitting. The cleaned dataset contains 325,423 rows. Using the same 80/10/10 stratified split with seed 42 produced the authoritative frozen splits recorded in `data_splits/split_metadata.json`:

- train: 260,338 rows
- validation: 32,542 rows
- test: 32,543 rows

The regenerated splits preserve the approximately 67.4% privacy-sensitive / 32.6% safe class balance and contain no exact duplicate text overlap across split boundaries.

**Text length:**  
Privacy-sensitive examples were slightly longer on average than safe examples. Length may provide supporting signal, but it is not sufficient by itself to identify PII.

**Placeholder tokens:**  
Bracket-style placeholder tokens appeared in both classes and were more common in the safe class in this check. Placeholder presence alone therefore does not determine the binary label.

**HTML-like formatting:**  
HTML-like formatting appeared in both classes but was more common in privacy-sensitive examples: approximately 16.9% of safe examples and 26.0% of privacy-sensitive examples. This may provide useful signal, but it is also a dataset limitation because models may learn formatting patterns in addition to PII context.

#### Final takeaway

The EDA directly changed the modeling workflow. Cleaning was completed before the final split, eliminated exact duplicate overlap across train, validation, and test, and preserved the original class balance. The remaining dataset characteristics—multilingual text, class imbalance, placeholder artifacts, structured numeric strings, and HTML-like formatting—were carried forward into feature engineering, model selection, and error analysis.
